# StockWise — Inventory and Stock-Out Analysis

This notebook explores the reproducible synthetic StockWise dataset. It covers data generation, cleaning, quality checks, inventory KPIs, supplier delivery performance, stock movements, and chart generation.

**Dataset note:** all records are synthetic teaching data generated with a fixed random seed. They do not describe a real business.

Run the notebook from the repository root or open it in Jupyter; paths are resolved automatically.

## 1. Import libraries and locate the project
The next cell imports the analysis packages and finds the repository root so the notebook can be run from either the root folder or `notebooks/`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = Path.cwd()
if not (BASE_DIR / "data" / "raw").exists():
    BASE_DIR = BASE_DIR.parent

RAW_DIR = BASE_DIR / "data" / "raw"
CLEAN_DIR = BASE_DIR / "data" / "clean"
REPORT_DIR = BASE_DIR / "reports"
CHART_DIR = REPORT_DIR / "charts"

print("Project directory:", BASE_DIR)
print("Python:", sys.version.split()[0])

## 2. Generate and clean the source data
The raw data generator uses a fixed seed. The cleaning script writes separate clean CSV files and a quality log; it does not overwrite the raw inputs.

In [ ]:
subprocess.run([sys.executable, str(BASE_DIR / "src" / "01_generate_data.py")], check=True)
subprocess.run([sys.executable, str(BASE_DIR / "src" / "02_clean_data.py")], check=True)

products = pd.read_csv(CLEAN_DIR / "products.csv")
suppliers = pd.read_csv(CLEAN_DIR / "suppliers.csv")
inventory = pd.read_csv(CLEAN_DIR / "inventory.csv")
purchase_orders = pd.read_csv(
    CLEAN_DIR / "purchase_orders.csv",
    parse_dates=["order_date", "expected_delivery_date", "actual_delivery_date"]
)
stock_movements = pd.read_csv(
    CLEAN_DIR / "stock_movements.csv",
    parse_dates=["movement_date"]
)

print("Products:", products.shape)
print("Suppliers:", suppliers.shape)
print("Inventory:", inventory.shape)
print("Purchase orders:", purchase_orders.shape)
print("Stock movements:", stock_movements.shape)

## 3. Review the data-quality log
The log records each detected issue, the correction rule, and the number of affected rows. The cleaned files are checked for duplicate primary keys and missing key fields.

In [ ]:
quality_log = pd.read_csv(REPORT_DIR / "cleaning_log.csv")
display(quality_log)

key_columns = {
    "products": ("product_id", products),
    "suppliers": ("supplier_id", suppliers),
    "inventory": ("inventory_id", inventory),
    "purchase_orders": ("order_id", purchase_orders),
    "stock_movements": ("movement_id", stock_movements),
}
quality_checks = []
for table_name, (key, frame) in key_columns.items():
    quality_checks.append({
        "dataset": table_name,
        "unique_primary_key": frame[key].is_unique,
        "missing_primary_key": int(frame[key].isna().sum()),
    })
display(pd.DataFrame(quality_checks))

## 4. Calculate inventory KPIs
Business rules: stock-out means available quantity is zero or below; low stock means available quantity is positive and at or below the reorder level; inventory value equals available quantity multiplied by unit cost.

In [ ]:
stock = inventory.merge(products, on="product_id", how="left", validate="one_to_one")
stock["stock_status"] = np.select(
    [
        stock["available_quantity"] <= 0,
        (stock["available_quantity"] > 0) &
        (stock["available_quantity"] <= stock["reorder_level"])
    ],
    ["Stock-out", "Low stock"],
    default="Healthy"
)
stock["inventory_value"] = stock["available_quantity"] * stock["unit_cost"]

kpis = pd.DataFrame({
    "Metric": [
        "Products", "Stock-out products", "Low-stock products",
        "Healthy products", "Total available units", "Inventory value"
    ],
    "Value": [
        stock["product_id"].nunique(),
        int((stock["stock_status"] == "Stock-out").sum()),
        int((stock["stock_status"] == "Low stock").sum()),
        int((stock["stock_status"] == "Healthy").sum()),
        int(stock["available_quantity"].sum()),
        round(stock["inventory_value"].sum(), 2),
    ]
})
display(kpis)

## 5. Compare stock levels and inventory value by category
The category summary highlights where stock and recorded inventory value are concentrated. The `Unclassified` category is retained because the source intentionally includes a missing category for cleaning practice.

In [ ]:
category_summary = (
    stock.groupby("category", as_index=False)
    .agg(
        products=("product_id", "nunique"),
        available_units=("available_quantity", "sum"),
        stockout_products=("stock_status", lambda x: int((x == "Stock-out").sum())),
        low_stock_products=("stock_status", lambda x: int((x == "Low stock").sum())),
        inventory_value=("inventory_value", "sum"),
    )
    .sort_values("inventory_value", ascending=False)
)
display(category_summary)

## 6. Analyse supplier delivery performance
Lead time is the calendar-day difference between actual delivery and order date. A delivery is delayed when its actual date is later than its expected date.

In [ ]:
purchase_orders["lead_time_days"] = (
    purchase_orders["actual_delivery_date"] - purchase_orders["order_date"]
).dt.days
purchase_orders["is_delayed"] = (
    purchase_orders["actual_delivery_date"] > purchase_orders["expected_delivery_date"]
)
supplier_summary = (
    purchase_orders.groupby("supplier_id", as_index=False)
    .agg(
        order_count=("order_id", "count"),
        average_lead_time_days=("lead_time_days", "mean"),
        delayed_orders=("is_delayed", "sum"),
    )
    .merge(suppliers[["supplier_id", "supplier_name"]], on="supplier_id", how="left")
    .sort_values("average_lead_time_days", ascending=False)
)
display(supplier_summary.round(2))

## 7. Examine stock movements over time
Incoming and outgoing quantities are aggregated by month. These are movement totals, not direct daily stock-on-hand observations.

In [ ]:
stock_movements["month"] = stock_movements["movement_date"].dt.to_period("M").astype(str)
monthly_movements = (
    stock_movements.pivot_table(
        index="month", columns="movement_type", values="quantity",
        aggfunc="sum", fill_value=0
    )
    .reset_index()
)
display(monthly_movements)

## 8. Generate the static charts
The chart script creates seven PNGs in `reports/charts/`. The estimated stock-out trend reconstructs daily balances from opening stock and movements, so it is an estimate rather than a measured daily inventory history.

In [ ]:
subprocess.run(
    [sys.executable, str(BASE_DIR / "src" / "04_analysis_charts.py")],
    check=True
)
chart_files = sorted(CHART_DIR.glob("*.png"))
print("Charts generated:", len(chart_files))
for chart in chart_files:
    print(chart.name, f"({chart.stat().st_size:,} bytes)")

## 9. Key findings from the generated dataset
The figures below are calculated from the cleaned data each time the notebook runs. They are descriptive results from synthetic records, not claims about a real retailer.

In [ ]:
stockout_count = int((stock["stock_status"] == "Stock-out").sum())
low_stock_count = int((stock["stock_status"] == "Low stock").sum())
delayed_count = int(purchase_orders["is_delayed"].sum())
average_lead = purchase_orders["lead_time_days"].mean()
top_value_category = category_summary.iloc[0]
top_stockout_category = (
    stock.loc[stock["stock_status"] == "Stock-out"]
    .groupby("category").size().sort_values(ascending=False)
)
print(f"Products analysed: {stock['product_id'].nunique()}")
print(f"Stock-out products: {stockout_count}")
print(f"Low-stock products: {low_stock_count}")
print(f"Inventory value: {stock['inventory_value'].sum():,.2f} dataset currency units")
print(f"Average supplier lead time: {average_lead:.2f} calendar days")
print(f"Delayed purchase orders: {delayed_count} of {len(purchase_orders)}")
print(f"Highest inventory-value category: {top_value_category['category']}")
if not top_stockout_category.empty:
    print(f"Category with most stock-outs: {top_stockout_category.index[0]} ({int(top_stockout_category.iloc[0])} products)")

## 10. Interpretation and limitations

- Prioritise products classified as stock-out or low stock for a manual replenishment review.
- Use supplier lead time and delayed-order counts to compare delivery reliability; a high average lead time alone does not explain every delay.
- Review category-level inventory value alongside stock availability so that value is not mistaken for service level.
- The data is synthetic, uses a single inventory snapshot, and has no real sales demand or operational costs. The reconstructed stock-out trend is based on an assumed opening balance plus dated movements.
- The analysis does not forecast demand or automatically recommend purchase quantities.

**Conclusion:** StockWise demonstrates a reproducible data preparation and analysis workflow. Business decisions should be validated with real operational data before use.